In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

load_dotenv()

model = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [5]:
def create_outline(state: BlogState) -> BlogState:
    title = state['title']

    prompt = f'Generate a detailed outline for a blog on the topic: {title}'

    outline = model.invoke(prompt).text

    state['outline'] = outline
    return state

In [6]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the following outline \n {outline}'

    content = model.invoke(prompt).text

    state['content'] = content
    return state

In [7]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

initial_state = {'title': 'Rise of AI in India'}
final_state = workflow.invoke(initial_state)

print(final_state)
print(final_state['outline'])   # the generated outline
print(final_state['content'])   # the full blog

{'title': 'Rise of AI in India', 'outline': 'Here is a comprehensive and detailed outline for a blog post on the **"Rise of AI in India."**\n\n---\n\n# Blog Title Ideas:\n* **Option 1:** The Silicon Subcontinent: How AI is Transforming India’s Digital Future\n* **Option 2:** From IT Hub to AI Powerhouse: The Rise of Artificial Intelligence in India\n* **Option 3:** India’s AI Revolution: Key Drivers, Breakthroughs, and the Road Ahead\n* **Option 4:** #AIforAll: How India is Building the World’s Most Inclusive AI Ecosystem\n\n---\n\n## I. Introduction\n* **The Hook:** A striking statistic or real-world example (e.g., how an Indian farmer uses a voice-based AI bot in his native dialect to check crop prices, or India\'s projected $500 billion contribution from AI to its GDP by 2026).\n* **Context:** India’s historical transition from a global IT outsourcing back-office to a front-runner in deep-tech innovation.\n* **Thesis Statement:** Driven by massive data availability, government polic